# Context Engineering for RAG

> **Retrieving the right information is only half the problem. We also need to construct the right context for the LLM.**

Our RAG pipeline has become:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Retrieval
    ↓
Hybrid Retrieval
    ↓
Reranking
```

At this point we have a ranked set of potentially useful chunks.

But we still have a critical question:

> **What exactly should we give to the LLM?**

A naive implementation might simply concatenate every retrieved chunk:

```text
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
    ↓
LLM
```

That can work for simple examples.

Production RAG requires more deliberate context construction.

This notebook introduces **context engineering**: selecting, organizing, formatting, and controlling the information that reaches the generation model.

## What We'll Learn

By the end of this tutorial, you'll understand:

- Retrieval results vs. model context
- Why retrieving relevant chunks is not enough
- Context selection
- Context ordering
- Redundancy and duplicate information
- Context length
- Context compression
- Source metadata and citations
- Handling conflicting evidence
- Building a context assembly function
- How context engineering fits into the complete RAG pipeline

The central idea is:

```text
Retrieval finds candidates.
Context engineering decides what the LLM actually sees.
```

# 1. Retrieval Is Not Context

This distinction is fundamental.

Our retriever might return:

```text
Top 10 chunks
```

But that does not mean all ten should automatically be passed to the LLM.

Think of the system as two stages:

```text
                    Query
                      ↓
                  Retrieval
                      ↓
               Candidate Chunks
                      ↓
             Context Engineering
                      ↓
                Final Context
                      ↓
                     LLM
```

Retrieval answers:

> **What information might be useful?**

Context engineering answers:

> **What information should the model actually receive, and how should it be presented?**

# 2. Why Naive Concatenation Can Fail

Suppose retrieval returns:

```text
Chunk A → directly answers the question
Chunk B → related background
Chunk C → duplicate of A
Chunk D → unrelated information
Chunk E → outdated policy
```

If we concatenate everything:

```text
A + B + C + D + E
```

we have created a noisy context.

The LLM now has to determine:

- Which information matters?
- Which information is duplicated?
- Which information conflicts?
- Which source is authoritative?
- Which information answers the actual question?

We can make this easier by constructing the context deliberately.

# 3. A Simple Example Corpus

Let's create a small corpus that contains relevant, redundant, and potentially conflicting information.

In [1]:
documents = [
    {
        "id": "refund-policy",
        "text": "Customers can request a refund within 30 days of purchase.",
        "source": "refund_policy.md",
        "date": "2026-01-10",
    },
    {
        "id": "refund-process",
        "text": "Approved refunds are normally processed within 7 business days.",
        "source": "refund_process.md",
        "date": "2026-01-15",
    },
    {
        "id": "refund-policy-duplicate",
        "text": "Customers can request a refund within 30 days of purchase.",
        "source": "customer_faq.md",
        "date": "2026-02-01",
    },
    {
        "id": "old-refund-policy",
        "text": "Customers can request a refund within 14 days of purchase.",
        "source": "archived_policy.md",
        "date": "2025-03-01",
    },
    {
        "id": "shipping",
        "text": "Standard shipping normally takes between 3 and 5 business days.",
        "source": "shipping.md",
        "date": "2026-01-20",
    },
]

# 4. Represent Retrieved Results

A production retrieval pipeline should preserve useful metadata.

For example:

```text
Document ID
Text
Retrieval score
Source
Date
Metadata
```

Let's create some example ranked results.

In [2]:
retrieved_results = [
    {
        **documents[0],
        "retrieval_score": 0.91,
    },
    {
        **documents[2],
        "retrieval_score": 0.88,
    },
    {
        **documents[1],
        "retrieval_score": 0.84,
    },
    {
        **documents[3],
        "retrieval_score": 0.79,
    },
    {
        **documents[4],
        "retrieval_score": 0.55,
    },
]

for rank, result in enumerate(retrieved_results, start=1):
    print(f"Rank {rank}: {result['id']}")
    print(f"Score: {result['retrieval_score']:.2f}")
    print(result["text"])
    print(f"Source: {result['source']}")
    print("---")

Rank 1: refund-policy
Score: 0.91
Customers can request a refund within 30 days of purchase.
Source: refund_policy.md
---
Rank 2: refund-policy-duplicate
Score: 0.88
Customers can request a refund within 30 days of purchase.
Source: customer_faq.md
---
Rank 3: refund-process
Score: 0.84
Approved refunds are normally processed within 7 business days.
Source: refund_process.md
---
Rank 4: old-refund-policy
Score: 0.79
Customers can request a refund within 14 days of purchase.
Source: archived_policy.md
---
Rank 5: shipping
Score: 0.55
Standard shipping normally takes between 3 and 5 business days.
Source: shipping.md
---


# 5. Context Selection

The first context-engineering decision is:

> **Which retrieved results should be included?**

A simple strategy is to select the top `k` results after reranking.

For example:

```text
20 retrieved candidates
        ↓
Reranking
        ↓
Top 5
        ↓
Context
```

But the number five is not a law.

The appropriate context size depends on:

- Query complexity
- Chunk size
- Model context window
- Redundancy
- Retrieval quality
- Latency
- Cost
- Expected answer length

The objective is not:

> **Include as much information as possible.**

It is:

> **Include enough high-quality evidence to answer the question reliably.**

# 6. Context Ordering

Ordering can matter.

Suppose we have:

```text
Chunk A — direct answer
Chunk B — supporting evidence
Chunk C — background
Chunk D — another supporting source
```

We need a policy for ordering them.

A simple approach is to preserve reranker order:

```text
Highest relevance
      ↓
...
      ↓
Lowest relevance
```

Another approach might place the most authoritative source first.

Another might group information by topic.

The correct strategy depends on the model and application.

The important point is:

> **Context order is an engineering decision, not an accident.**

# 7. Remove Redundancy

Our example corpus contains:

```text
refund-policy
refund-policy-duplicate
```

Both communicate essentially the same information.

Passing both may waste context space.

Redundancy can occur because:

- Multiple documents contain the same policy
- Overlapping chunks were created
- Different sources repeat the same information
- Retrieval returns neighboring chunks with overlapping text

We can detect exact duplicates easily.

In [4]:
def remove_exact_duplicates(results):
    seen = set()
    unique_results = []

    for result in results:
        text = result["text"].strip()

        if text in seen:
            continue

        seen.add(text)
        unique_results.append(result)

    return unique_results

unique_results = remove_exact_duplicates(retrieved_results)

for result in unique_results:
    print(result["id"], "→", result["text"])

refund-policy → Customers can request a refund within 30 days of purchase.
refund-process → Approved refunds are normally processed within 7 business days.
old-refund-policy → Customers can request a refund within 14 days of purchase.
shipping → Standard shipping normally takes between 3 and 5 business days.


Exact duplicate removal is useful, but it is only the simplest case.

Two chunks can contain nearly identical information without having identical text.

More advanced systems can use:

- Similarity-based deduplication
- Clustering
- MMR-style selection
- Semantic compression

The trade-off is that more sophisticated processing adds computation and complexity.

# 8. Maximum Marginal Relevance

One common way to balance relevance and diversity is **Maximum Marginal Relevance (MMR)**.

The intuition is:

```text
Select something relevant
+
Penalize information that is too similar to what we already selected
```

Conceptually:

```text
MMR
=
Relevance
-
Redundancy
```

This can help when the retriever returns many highly similar chunks.

The goal is not simply to find the five most similar chunks.

It is to find five chunks that are:

> **relevant to the query and useful together.**

# 9. Context Length

Every model has practical context limitations.

Even when a model supports a very large context window, more context is not automatically better.

More context can mean:

- Higher latency
- Higher cost
- More irrelevant information
- More competition between evidence
- More difficult source attribution

So we should think of context as a resource.

```text
Context budget
      ↓
Relevant evidence
      +
Necessary instructions
      +
Conversation state
```

The objective is to use the available context budget efficiently.

# 10. A Simple Context Budget

We can start with a simple character-based budget for demonstration purposes.

This is not a substitute for tokenizer-based accounting in production, but it makes the idea easy to see.

In [5]:
def select_by_character_budget(results, max_chars=800):
    selected = []
    total_chars = 0

    for result in results:
        text = result["text"]

        if total_chars + len(text) > max_chars:
            continue

        selected.append(result)
        total_chars += len(text)

    return selected

budgeted_results = select_by_character_budget(
    unique_results,
    max_chars=250
)

for result in budgeted_results:
    print(result["id"])
    print(result["text"])
    print("---")

refund-policy
Customers can request a refund within 30 days of purchase.
---
refund-process
Approved refunds are normally processed within 7 business days.
---
old-refund-policy
Customers can request a refund within 14 days of purchase.
---
shipping
Standard shipping normally takes between 3 and 5 business days.
---


In a real application, use the tokenizer associated with the generation model when context size must be controlled precisely.

The principle remains the same:

```text
Available context budget
        ↓
Prioritize useful evidence
```

# 11. Metadata and Source Attribution

Context should not necessarily contain only raw text.

Useful metadata can help the model and the application understand where information came from.

For example:

```text
[Source: refund_policy.md]
Customers can request a refund within 30 days...
```

This can support citation generation and source attribution.

Metadata can include:

- Source name
- Document ID
- Page number
- Section
- Date
- Author
- URL
- Access permissions

The exact metadata depends on the application.

# 12. Build a Context Formatter

Let's create a simple formatter that preserves source information.

In [6]:
def format_context(results):
    context_blocks = []

    for index, result in enumerate(results, start=1):
        block = (
            f"[Source {index}]\n"
            f"Document: {result['source']}\n"
            f"Date: {result['date']}\n"
            f"Content: {result['text']}"
        )

        context_blocks.append(block)

    return "\n\n".join(context_blocks)

context = format_context(budgeted_results)

print(context)

[Source 1]
Document: refund_policy.md
Date: 2026-01-10
Content: Customers can request a refund within 30 days of purchase.

[Source 2]
Document: refund_process.md
Date: 2026-01-15
Content: Approved refunds are normally processed within 7 business days.

[Source 3]
Document: archived_policy.md
Date: 2025-03-01
Content: Customers can request a refund within 14 days of purchase.

[Source 4]
Document: shipping.md
Date: 2026-01-20
Content: Standard shipping normally takes between 3 and 5 business days.


Now our context has a structure the generation stage can work with:

```text
[Source 1]
Document: ...
Date: ...
Content: ...

[Source 2]
Document: ...
Date: ...
Content: ...
```

# 13. Conflicting Information

One of the hardest context problems is conflicting evidence.

Our corpus contains:

```text
Current policy:
30 days

Archived policy:
14 days
```

A naive system might pass both to the LLM without explanation.

That creates ambiguity.

A better pipeline should preserve metadata that helps distinguish sources:

```text
Current policy
2026-01-10
30 days

Archived policy
2025-03-01
14 days
```

Then we can apply application-specific rules.

For example:

```text
Prefer current policy
over archived policy
```

But be careful:

> **The RAG system should not invent authority rules that the application hasn't defined.**

If document freshness or source authority matters, those rules should be explicit and testable.

# 14. Context Is Not the Place to Hide Retrieval Problems

Context engineering cannot compensate for everything.

Suppose the correct document was never retrieved.

Then:

```text
Retrieval
    ↓
No correct evidence
    ↓
Context engineering
    ↓
Still no correct evidence
```

Likewise, if the retrieval stage returns only irrelevant chunks, formatting them more nicely won't solve the underlying problem.

The layers have different responsibilities:

```text
Retrieval
→ Find useful evidence

Reranking
→ Prioritize useful evidence

Context engineering
→ Construct useful model input

Generation
→ Produce the answer
```

Keeping these responsibilities separate makes debugging much easier.

# 15. Build a Context Assembly Pipeline

Let's combine our simple techniques into one function.

Our demonstration pipeline will:

1. Remove exact duplicates.
2. Preserve ranking order.
3. Apply a context budget.
4. Format source metadata.

In [7]:
def build_context(
    results,
    max_chars=1000,
):
    results = remove_exact_duplicates(results)

    results = select_by_character_budget(
        results,
        max_chars=max_chars,
    )

    return format_context(results)

final_context = build_context(
    retrieved_results,
    max_chars=1000,
)

print(final_context)

[Source 1]
Document: refund_policy.md
Date: 2026-01-10
Content: Customers can request a refund within 30 days of purchase.

[Source 2]
Document: refund_process.md
Date: 2026-01-15
Content: Approved refunds are normally processed within 7 business days.

[Source 3]
Document: archived_policy.md
Date: 2025-03-01
Content: Customers can request a refund within 14 days of purchase.

[Source 4]
Document: shipping.md
Date: 2026-01-20
Content: Standard shipping normally takes between 3 and 5 business days.


This is deliberately simple.

A production implementation may also include:

- Semantic deduplication
- MMR
- Token-based budgeting
- Source authority rules
- Metadata filtering
- Context compression
- Citation tracking
- Access-control filtering
- Conversation history management

# 16. Context Compression

Sometimes a retrieved chunk contains useful information plus a lot of irrelevant material.

Instead of passing the entire chunk, we can attempt to compress it.

Conceptually:

```text
Retrieved chunk
      ↓
Extract relevant information
      ↓
Shorter context
```

For example:

```text
Original:
A long policy containing many sections...

Compressed:
Refunds may be requested within 30 days.
```

Compression can reduce context size.

But it introduces another transformation step and therefore another possible source of information loss.

This means compression should also be evaluated.

# 17. Context Engineering and Citations

If the application needs citations, citation information should survive the retrieval-to-generation pipeline.

A useful representation is:

```text
Evidence
  ├── text
  ├── document_id
  ├── source
  ├── page
  └── section
```

Then the generation layer can associate claims with evidence.

This is preferable to trying to reconstruct the source after the answer has already been generated.

The general principle is:

> **Preserve provenance throughout the pipeline.**

# 18. Context Engineering and Access Control

Context engineering also has a security dimension.

Suppose a user is not authorized to see a document.

It should not reach the model merely because retrieval found it.

The safe architecture is:

```text
User
  ↓
Authorization
  ↓
Retrieval constrained by permissions
  ↓
Reranking
  ↓
Context
  ↓
LLM
```

Access control should therefore be enforced before sensitive content enters the model context.

This is especially important for enterprise RAG systems with multiple users or teams.

# 19. The Complete RAG Flow

We can now see how the stages fit together:

```text
                    Documents
                        ↓
                   Ingestion
                        ↓
                    Chunking
                        ↓
                   Embeddings
                        ↓
                 Vector Index
                        ↓
                  User Query
                        ↓
            Dense + Lexical Retrieval
                        ↓
                       RRF
                        ↓
                    Reranking
                        ↓
              Context Engineering
                        ↓
                Grounded Generation
                        ↓
                     Answer
```

Each stage solves a different problem.

That separation is one of the most important architectural ideas in production RAG.

# 20. Debugging Context Problems

When a generated answer is poor, inspect the pipeline in order.

Ask:

### 1. Was the correct document retrieved?

If not, investigate retrieval.

### 2. Was the correct document ranked highly enough?

If not, investigate reranking.

### 3. Was the relevant information included in the final context?

If not, investigate context selection.

### 4. Was the context ordered and formatted sensibly?

If not, investigate context construction.

### 5. Did the model correctly use the evidence?

If not, investigate generation and grounding.

This gives us a useful debugging chain:

```text
Retrieval
   ↓
Reranking
   ↓
Context
   ↓
Generation
```

Don't immediately blame the LLM.

# Key Takeaways

1. Retrieval results are not automatically the final context.
2. Context engineering decides what information the LLM actually receives.
3. More context is not necessarily better.
4. Context selection should balance relevance, coverage, redundancy, cost, and latency.
5. Context ordering is an engineering decision.
6. Duplicate and highly redundant information can waste context.
7. MMR is one approach for balancing relevance and diversity.
8. Context budgets should be controlled, preferably using the model's tokenizer in production.
9. Preserve source metadata and provenance throughout the pipeline.
10. Conflicting sources require explicit application rules rather than model guesswork.
11. Context engineering cannot recover evidence that retrieval never found.
12. Access control should prevent unauthorized documents from reaching the model.
13. Context compression can reduce input size but must itself be evaluated.
14. The goal is not maximum context; it is **useful, grounded context**.

The mental model is:

```text
Retrieve broadly
      ↓
Rank carefully
      ↓
Select deliberately
      ↓
Format clearly
      ↓
Generate from evidence
```

# What's Next?

We can now construct a deliberate context from our retrieved evidence.

The next question is:

> **How do we make the LLM answer from that context rather than simply producing a plausible answer?**

That takes us into **grounded generation**.

We'll look at:

- Prompting with retrieved evidence
- Grounded answers
- Source attribution
- Citations
- Abstention
- Insufficient evidence
- Hallucination
- Faithfulness
- Structured generation

The pipeline becomes:

```text
Retrieval
    ↓
Reranking
    ↓
Context Engineering
    ↓
Grounded Generation
    ↓
Answer
```